# Part 1 - full-test-split BLEU for the five trained checkpoints

The model notebooks score BLEU on a seeded 1,000-triplet sample of the test split (2,000 translations). This notebook re-scores every saved checkpoint on **all** retained test triplets, using the unchanged `evaluate_bleu` / `greedy_translations` from `shared/implementation.ipynb`. No training happens here.

It also reports BLEU on the subset of test triplets whose Vietnamese and Japanese sources never occur verbatim as a training source. The official dataset splits share some templated source strings, often with different English targets.

In [1]:
from pathlib import Path
import json, os
here=Path.cwd().resolve()
ROOT=next((p for p in [here,*here.parents,here/'part 1'] if (p/'shared'/'implementation.ipynb').exists()),None)
assert ROOT is not None, 'Set the notebook working directory inside part 1.'
for cell in json.loads((ROOT/'shared'/'implementation.ipynb').read_text())['cells']:
    if cell['cell_type']=='code' and 'definitions' in cell.get('metadata',{}).get('tags',[]):
        exec(compile(cell['source'],'shared/implementation.ipynb','exec'),globals())

In [2]:
BOOTSTRAP=int(os.environ.get('PART1_BLEU_BOOTSTRAP','200'))
test_rows=json.loads((ROOT/'shared'/'data'/'test'/'raw.json').read_text())
train_rows=json.loads((ROOT/'shared'/'data'/'train'/'raw.json').read_text())
train_sources={lang:{r[lang] for r in train_rows} for lang in LANGS}
seen=np.array([any(r[lang] in train_sources[lang] for lang in LANGS) for r in test_rows])
del train_rows
tok=Tokenizer.from_file(str(ROOT/'shared'/'tokenizer_clean.json'))
print('device:',DEVICE,'| test triplets:',len(test_rows),'| with a source seen verbatim in train:',int(seen.sum()))

device: cuda | test triplets: 24791 | with a source seen verbatim in train: 1097


In [3]:
metric=sacrebleu.metrics.BLEU(); results=[]
for model_id in range(1,6):
    path=ROOT/f'model {model_id}'/'full'/'checkpoint.pt'
    saved=torch.load(path,map_location='cpu',weights_only=False)
    assert saved['tokenizer_sha256']==sha(ROOT/'shared'/'tokenizer_clean.json')
    cfg=Config(**saved['config']); model=Transformer(cfg).to(DEVICE).eval(); model.load_state_dict(saved['model'])
    checkpoint_sha=sha(path); del saved
    bleu,records=evaluate_bleu(model,tok,cfg,split='test',nrows=len(test_rows),bootstrap=BOOTSTRAP)
    assert bleu['triplets']==len(test_rows)
    clean=[r for r in records if not seen[r['row']]]
    bleu['bleu_unseen_sources']=metric.corpus_score([r['hypothesis'] for r in clean],[[r['reference'] for r in clean]]).score
    bleu['translations_unseen_sources']=len(clean)
    for lang in LANGS:
        part=[r for r in clean if r['language']==lang]
        bleu[f'bleu_{lang}_unseen_sources']=metric.corpus_score([r['hypothesis'] for r in part],[[r['reference'] for r in part]]).score
    row={'model':model_id,'name':NAMES[model_id],'checkpoint_sha256':checkpoint_sha,**bleu}
    results.append(row); print(json.dumps({k:row[k] for k in ('model','bleu','bleu_vi','bleu_ja','bleu_unseen_sources','seconds')}))
    out=ROOT/f'model {model_id}'/'full'
    (out/'full_test_bleu.json').write_text(json.dumps(row,indent=2))
    with open(out/'full_test_predictions.jsonl','w',encoding='utf-8') as handle:
        for r in records: handle.write(json.dumps(r,ensure_ascii=False)+'\n')
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
(ROOT/'shared'/'full_test_bleu.json').write_text(json.dumps(results,indent=2))

{"model": 1, "bleu": 33.97810201299469, "bleu_vi": 39.066982012855746, "bleu_ja": 28.792525850995446, "bleu_unseen_sources": 33.55827604481658, "seconds": 916.5184719949029}


{"model": 2, "bleu": 33.51471309930346, "bleu_vi": 38.499119423197385, "bleu_ja": 28.434586783020602, "bleu_unseen_sources": 33.09073580234199, "seconds": 1877.4446838968433}


{"model": 3, "bleu": 33.556974642836444, "bleu_vi": 38.616230401639484, "bleu_ja": 28.396989914873615, "bleu_unseen_sources": 33.12763960744753, "seconds": 1959.7248376708012}


{"model": 4, "bleu": 33.60124664140663, "bleu_vi": 38.64791250700646, "bleu_ja": 28.45459059539611, "bleu_unseen_sources": 33.18551081500865, "seconds": 1932.0093240109272}


{"model": 5, "bleu": 34.130185302252464, "bleu_vi": 39.02589882399936, "bleu_ja": 29.144152689065514, "bleu_unseen_sources": 33.69119447999199, "seconds": 1995.2428130940534}


4255

In [4]:
table=pd.DataFrame(results)[['model','name','triplets','bleu','bleu_ci95','bleu_vi','bleu_ja','bleu_unseen_sources','translations_unseen_sources','generation_cap_vi','generation_cap_ja','seconds','signature']]
display(table)
display({'application/json':{'status':'full test BLEU completed','results':[{k:v for k,v in r.items()} for r in results]}},raw=True)

,model,name,triplets,bleu,bleu_ci95,bleu_vi,bleu_ja,bleu_unseen_sources,translations_unseen_sources,generation_cap_vi,generation_cap_ja,seconds,signature
0,1,Dense two-layer MLP,24791,33.978102,"[33.711919720326485, 34.31288097748315]",39.066982,28.792526,33.558276,47388,31,60,916.518472,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...
1,2,4 experts; top 1; equal total parameters,24791,33.514713,"[33.262787354083564, 33.866494157001256]",38.499119,28.434587,33.090736,47388,22,41,1877.444684,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...
2,3,4 experts; top 2; equal total parameters,24791,33.556975,"[33.29968279205873, 33.91069770764348]",38.616230,28.396990,33.127640,47388,24,34,1959.724838,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...
3,4,1 always-active shared + 3 routed; top 1; equa...,24791,33.601247,"[33.3433930010359, 33.92648070176209]",38.647913,28.454591,33.185511,47388,23,44,1932.009324,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...
4,5,4 experts; top 2; equal active parameters,24791,34.130185,"[33.847983088002565, 34.47211372792926]",39.025899,29.144153,33.691194,47388,20,27,1995.242813,nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v...
